# 01 — Introduction (CSS)

A guided tour of the CSS builder (level 1). Each section is a
small self-contained example showing the code, the rendered CSS
source, and a live HTML demo that applies the CSS to a few
hand-written elements.

The general lifecycle (`create()` / `build()` / `render()`) and
the `source` / `built` distinction are covered in the HTML
introduction notebook — they work identically here.

**Topics (primary example, mirrored by `01_introduction.py`):**
1. Hello CSS — fragment with a single rule.
2. Stylesheet with multiple rules.
3. Selectors built from kwargs.
4. CSS variables — `cssvar` inside `:root`.
5. Comments — short inline vs long block.

**Notebook-only extras (after the divider):**
- Fluent chaining with `._`.

## 1. Hello CSS

A CSS document is a subclass of `CssBuilderHandler` that
implements `main(self, root)`. The simplest unit is a fragment:
one `rule(...)` with property kwargs, plus at least one child
`selector(...)` that names where the rule applies.

In [ ]:
from IPython.display import HTML, Code

from genro_builders.contrib.css import CssBuilderHandler


class HelloCss(CssBuilderHandler):
    def main(self, root):
        r = root.rule(color="white", background_color="#3498db", padding="12px")
        r.selector(_class="card")


page = HelloCss()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(f"<style>{css}</style><div class='card'>Hello CSS</div>")

## 2. Stylesheet with multiple rules

Wrap rules in a `stylesheet()` container when you want a full
document. A `stylesheet` is optional — fragments (a single rule
at the bag root) also work.

In [ ]:
class Theme(CssBuilderHandler):
    def main(self, root):
        sheet = root.stylesheet()

        r1 = sheet.rule(color="#3498db", font_weight="bold")
        r1.selector(_class="primary")

        r2 = sheet.rule(color="#e74c3c", font_weight="bold")
        r2.selector(_class="danger")

        r3 = sheet.rule(padding="8px", border_radius="4px")
        r3.selector(_class="primary")
        r3.selector(_class="danger")


page = Theme()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<span class='primary'>Primary</span> "
    "<span class='danger'>Danger</span>"
)

## 3. Selectors built from kwargs

`selector(...)` accepts structured kwargs:

| Kwarg | What it adds |
| --- | --- |
| `tag="div"` | element name (`div`) |
| `id="main"` | id (`#main`) |
| `_class="card"` | one class (`.card`) |
| `classes=["a","b"]` | compound classes (`.a.b`) |
| `attr={"type":"text"}` | attribute selector (`[type="text"]`) |
| `raw="> .icon"` | opaque suffix appended with a leading space |

Structured kwargs are validated (a class name with a space, a
dot, or a combinator raises a `ValueError`). `raw` is the
escape hatch for combinators, functional pseudo-classes, and
anything else not covered by the structured form.

In [ ]:
class Selectors(CssBuilderHandler):
    def main(self, root):
        sheet = root.stylesheet()

        # compound classes via list
        r1 = sheet.rule(border="2px solid #3498db", padding="8px")
        r1.selector(classes=["card", "highlighted"])

        # tag + attribute selector
        r2 = sheet.rule(background_color="#f8f8f8", padding="4px")
        r2.selector(tag="input", attr={"type": "text"})

        # class + pseudo attached
        r3 = sheet.rule(color="#3498db", cursor="pointer")
        r3.selector(_class="card:hover")

        # raw for combinator
        r4 = sheet.rule(font_weight="bold")
        r4.selector(_class="card", raw="> .title")


page = Selectors()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='card highlighted'>"
    "  <div class='title'>Card title (bold via combinator)</div>"
    "  <input type='text' value='input with grey bg'/>"
    "</div>"
)

## 4. CSS variables — cssvar inside :root

Declare CSS custom properties (`--name: value;`) with
`cssvar(name, value=...)`. They are children of a rule (typically
the `:root` rule). Consume them in property values via the
standard `var(--name)` string.

In [ ]:
class Themed(CssBuilderHandler):
    def main(self, root):
        sheet = root.stylesheet()

        rt = sheet.rule()
        rt.selector(raw=":root")
        rt.cssvar("brand", value="#3498db")
        rt.cssvar("spacing", value="8px")
        rt.cssvar("radius", value="6px")

        r = sheet.rule(
            background_color="var(--brand)",
            color="white",
            padding="var(--spacing)",
            border_radius="var(--radius)",
        )
        r.selector(_class="brand-card")


page = Themed()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='brand-card'>Themed via CSS variables</div>"
)

## 5. Comments — short inline vs long block

Any element accepts a `comment="..."` kwarg. The renderer picks
the position based on length:

- `len(comment) <= 60` → inline `/* ... */` at the end of the
  last property line.
- `len(comment) > 60` → block `/* ... */` on its own line above
  the element.

The threshold is a deliberate choice: short comments stay close
to what they annotate without breaking the visual flow; long
ones get their own line because they would push the property
line past readable width.

In [ ]:
class Commented(CssBuilderHandler):
    def main(self, root):
        sheet = root.stylesheet()

        # Short comment — inline.
        r1 = sheet.rule(color="red", comment="error state")
        r1.selector(_class="error")

        # Long comment — block above.
        r2 = sheet.rule(
            display="grid",
            grid_template_columns="repeat(auto-fit, minmax(200px, 1fr))",
            comment=(
                "Auto-fit grid: cards reflow without media queries. "
                "Used for the dashboard summary blocks."
            ),
        )
        r2.selector(_class="dashboard")

        # Comment on a cssvar.
        rt = sheet.rule()
        rt.selector(raw=":root")
        rt.cssvar("brand", value="#3498db", comment="main brand color")


page = Commented()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='error'>An error message</div>"
    "<div class='dashboard'>"
    "  <div style='background:#eee;padding:8px;'>Card 1</div>"
    "  <div style='background:#eee;padding:8px;'>Card 2</div>"
    "  <div style='background:#eee;padding:8px;'>Card 3</div>"
    "</div>"
)

---

## Beyond the primary example

The sections above mirror exactly what `01_introduction.py` produces. What follows is **notebook-only didactic material**: patterns and idioms worth showing in tutorial form but intentionally not included in the standalone script, to keep the script focused on the core example.

## Extra — fluent chaining with `._`

Every leaf element call (`selector(...)`, `cssvar(...)`) returns
the leaf node. Genro-bag exposes `._` on a node as a reference
to the bag that contains it — i.e. the rule's body. Chaining
`._` after a leaf call lets you keep adding siblings without
breaking the expression:

```python
root.rule(color="red", font_size="14px") \
    .selector(_class="card")._ \
    .selector(_class="panel")._ \
    .cssvar("primary", value="#3498db")
```

This is just sugar — the non-chained form (assign the rule to
a variable and call methods on it) stays valid and is often
clearer when the body is large. Pick whichever reads better
in context.

In [ ]:
class Fluent(CssBuilderHandler):
    def main(self, root):
        # Whole rule expressed as one chain.
        root.rule(color="white", background_color="#3498db", padding="12px") \
            .selector(_class="card")._ \
            .selector(_class="panel")._ \
            .selector(_class="dialog")


page = Fluent()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='card'>card</div> "
    "<div class='panel'>panel</div> "
    "<div class='dialog'>dialog</div>"
)

## 7. Rule nesting

A `rule` can contain other `rule` children. Nested rules are
emitted as native CSS Nesting (browsers 2023+); the browser
flattens them at parse time.

Two composition rules to remember when authoring:

- a nested selector that does **not** start with `&` is a
  **descendant** of the parent (`.title` inside `.card` resolves
  to `.card .title`);
- a nested selector that **starts with `&`** binds tightly to the
  parent (`&:hover` inside `.card` resolves to `.card:hover`).

Use `raw="..."` for any selector that involves `&` or a
combinator.

In [ ]:
class Nested(CssBuilderHandler):
    def main(self, root):
        sheet = root.stylesheet()

        card = sheet.rule(padding="8px", background_color="#fafafa")
        card.selector(_class="card")

        title = card.rule(font_size="18px", font_weight="bold")
        title.selector(_class="title")

        icon = card.rule(width="16px", color="#3498db")
        icon.selector(raw="& > .icon")

        hover = card.rule(background_color="#eef")
        hover.selector(raw="&:hover")


page = Nested()
page.create()
page.build()
css = page.render()

In [ ]:
Code(css, language='css')

In [ ]:
HTML(
    f"<style>{css}</style>"
    "<div class='card'>"
    "  <div class='title'>Title</div>"
    "  <span class='icon'>>></span>"
    "  Body of the card. Hover the block to see the background change."
    "</div>"
)